# 푸드코트 AI · 부분(확인) 시뮬레이션

본 실험(`extended_main_20261108_001`, 27개 점수표 × 5회)이 끝난 뒤, **고른 점수표만** 또는 **지정한 구간 안에서 새로 뽑은 가중치**로 AI를 다시 키워 보는 노트북입니다.

- 본 실험과 **같은 소스 버전(642cf41), 같은 설정, 같은 torch 빌드**를 씁니다. 바꾸는 것은 점수표와 반복 번호뿐입니다.
- 기존 점수표는 **새 반복 번호(5번 이후)** 로 돌립니다. 시드가 `(설정 시드, 점수표 id, 반복 번호)`로 정해지므로 원래 5회와 겹치지 않는 독립 반복입니다.
- 구간 샘플링은 λ(손님 몫), q(이익 vs 폐기), b(가격 부담 가중치), a_share(나머지 중 맛 비율 a/(a+c))의 범위를 정하면 그 안을 라틴 하이퍼큐브로 고르게 나눠 뽑습니다.
- 결과는 Drive `MyDrive/foodcourt_adaptive_sim/<실험 ID>`에 저장되고, 본 실험 폴더의 기존 반복과 합쳐 `partial_summary.csv`로 정리됩니다.
- 런타임이 끊기면 같은 실험 ID로 다시 실행하면 체크포인트부터 이어집니다.

In [ ]:
# 1. 소스 가져오기 · 설치 (본 실험과 같은 버전)
import os, sys, subprocess, json
from pathlib import Path
REPOSITORY = 'https://github.com/phj9187/foodcourt-adaptive-sim.git'
SOURCE_COMMIT = '642cf4180e54fe7102c5f160a2235f0abc017dfe'
PARTIAL_REPOSITORY = 'https://github.com/phj9187/foodcourt-partial-test.git'
WORK = Path('/content/foodcourt-adaptive-sim')
TOOL = Path('/content/foodcourt-partial-test')
if not WORK.exists():
    subprocess.run(['git','clone',REPOSITORY,str(WORK)],check=True)
subprocess.run(['git','-C',str(WORK),'fetch','-q','origin'],check=True)
subprocess.run(['git','-C',str(WORK),'checkout','-q','--detach',SOURCE_COMMIT],check=True)
if not TOOL.exists():
    subprocess.run(['git','clone',PARTIAL_REPOSITORY,str(TOOL)],check=True)
else:
    subprocess.run(['git','-C',str(TOOL),'pull','-q','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(WORK)],check=True)
if str(WORK/'src') not in sys.path:
    sys.path.insert(0,str(WORK/'src'))
# 본 실험과 같은 torch 빌드
_v=subprocess.run([sys.executable,'-c','import torch;print(torch.__version__)'],capture_output=True,text=True).stdout.strip()
if not _v.startswith('2.14.1+cpu'):
    subprocess.run([sys.executable,'-m','pip','install','-q','--force-reinstall','--no-deps','torch==2.14.1+cpu','--index-url','https://download.pytorch.org/whl/cpu'],check=True)
    _v=subprocess.run([sys.executable,'-c','import torch;print(torch.__version__)'],capture_output=True,text=True).stdout.strip()
print('시뮬레이션 소스:',subprocess.check_output(['git','-C',str(WORK),'rev-parse','HEAD'],text=True).strip())
print('부분 테스트 코드:',subprocess.check_output(['git','-C',str(TOOL),'rev-parse','HEAD'],text=True).strip())
print('torch:',_v,' CPU 코어:',os.cpu_count())

In [ ]:
# 2. Google Drive 연결 · 무엇을 돌릴지 정하기 (아직 학습하지 않음)
from google.colab import drive
import re
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/foodcourt_adaptive_sim')
REFERENCE = ROOT/'extended_main_20261108_001'   # 끝난 본 실험 (읽기만 함)
EXPERIMENT = 'partial_box_20261109_001'          # 이번 부분 테스트 폴더 (내용을 바꾸면 새 ID)
CONFIG = 'config/adaptive_extended.yaml'         # 본 실험과 동일

OBJECTIVES = 'lhs_0022,lhs_0019'  # 기존 점수표 다시 돌리기, 예: 'lhs_0022,lhs_0019' (없으면 '')
SAMPLE = 8               # 구간 안에서 새로 뽑을 가중치 개수 (0이면 안 뽑음)
RANGE = 'lambda=0.65..0.95,q=0.35..0.65,b=0..0.10,a_share=0..1'
SAMPLE_SEED = 20261109
REPS = '5'               # 반복 번호. 기존 점수표는 5 이후만 (0~4는 본 실험에 있음)
WORKERS = os.cpu_count() or 1

if not re.fullmatch(r'[A-Za-z0-9_-]+',EXPERIMENT):
    raise ValueError('실험 ID에는 영문, 숫자, 밑줄, 하이픈만 사용하세요.')
OUTPUT = ROOT/EXPERIMENT
OUTPUT.mkdir(parents=True,exist_ok=True)
ARGS=['--source',str(WORK),'--config',CONFIG,'--output',str(OUTPUT),'--reps',REPS,'--workers',str(WORKERS),
      '--sample',str(SAMPLE),'--range',RANGE,'--sample-seed',str(SAMPLE_SEED)]
if OBJECTIVES: ARGS+=['--objectives',OBJECTIVES]
if REFERENCE.exists(): ARGS+=['--reference',str(REFERENCE)]
else: print('본 실험 폴더가 없어 합산 없이 새 결과만 정리합니다:',REFERENCE)
plan=subprocess.run([sys.executable,str(TOOL/'partial_test.py'),*ARGS,'--dry-run'],cwd=WORK,capture_output=True,text=True)
print(plan.stdout); print(plan.stderr[-2000:])
plan.check_returncode()
jobs=plan.stdout.count('\n  ')*len([r for r in REPS.split(',') for _ in [0]])
print('저장 위치:',OUTPUT,' 병렬 작업 수:',WORKERS)

In [ ]:
# 3. 학습 · 테스트 실행 (같은 실험 ID로 다시 실행하면 이어서 진행)
from datetime import datetime, timezone
from kiosk_adaptive.study import atomic_json
log_path = OUTPUT/'run.log'
status_path = OUTPUT/'run_status.json'
def status(value,**extra):
    atomic_json(status_path,dict(status=value,updated_utc=datetime.now(timezone.utc).isoformat(),**extra))
command=[sys.executable,'-u',str(TOOL/'partial_test.py'),*ARGS]
status('running',command=command)
process=None
try:
    with log_path.open('a',encoding='utf-8',buffering=1) as log:
        process=subprocess.Popen(command,cwd=WORK,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            log.write(line)
            if line.startswith(('DONE','jobs=','lhs_','custom_','Traceback')) or 'Error' in line:
                print(line,end='')
        code=process.wait()
    status('complete' if code==0 else 'failed',exit_code=code)
    if code: raise RuntimeError(f'실행 실패: {code}. run.log를 확인하세요.')
except KeyboardInterrupt:
    if process is not None and process.poll() is None:
        process.terminate()
        try: process.wait(timeout=20)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
    status('interrupted')
    print('중단됨. 같은 실험 ID로 다시 실행하면 마지막 저장 지점부터 이어갑니다.')
except Exception as exc:
    status('failed',error=str(exc)); raise

In [ ]:
# 4. 결과 — 안정 조건, 반복 평균 (손님 점수는 100점 만점 기준 변화, 이익은 가게·영업일당 원)
import csv
path=OUTPUT/'partial_summary.csv'
if path.exists():
    rows=[r for r in csv.DictReader(path.open(encoding='utf-8-sig')) if r['scenario']=='stable' and r['group'] in ('new_only','pooled')]
    f=lambda v,d=2:'' if v in ('',None) else f'{float(v):+.{d}f}'
    print(f"{'점수표':24}{'묶음':10}{'n':>3}{'손님':>9}{'손님 95%':>20}{'이익(원)':>11}{'이익 95%':>22}{'F':>10}")
    for r in rows:
        print(f"{r['id']:24}{r['group']:10}{r['replicas']:>3}{f(r['customer_delta_per100']):>9}"
              f"{'['+f(r['customer_lo'])+', '+f(r['customer_hi'])+']' if r['customer_lo'] else '':>20}"
              f"{f(r['profit_delta_won'],0):>11}{'['+f(r['profit_lo'],0)+', '+f(r['profit_hi'],0)+']' if r['profit_lo'] else '':>22}{f(r['F_delta'],5):>10}")
    print('\n자세한 값:',OUTPUT/'partial_summary.json')
else:
    print('아직 끝난 작업이 없습니다. run.log를 확인하세요.')